# 01 - Investigar, crear guion y generar voz

Este notebook busca páginas públicas por tema con ScrapeGraphAI/DuckDuckGo (sin claves de redes), analiza una URL opcional y genera narración con OmniVoice. Guarda estos secretos en Colab: `API_BASE_URL` (HTTPS publico), `WORKER_API_TOKEN`, `SCRIPT_SPACE_URL` (Space privado de Qwen), `VOICE_STUDIO_URL`, `HF_TOKEN` (solo lectura de ambos Spaces), `S3_ENDPOINT_URL`, `S3_BUCKET`, `S3_ACCESS_KEY_ID` y `S3_SECRET_ACCESS_KEY`. `YOUTUBE_API_KEY` es opcional para señales de vistas por hora en YouTube.

El resultado sube el audio a R2 privado y crea automaticamente el trabajo de renderizado. El dashboard muestra el progreso de cada etapa.

**Modo worker continuo:** ejecuta las celdas desde arriba y deja activa la celda final. El dashboard local encola automáticamente los trabajos; esta sesión los toma mientras Colab permanezca conectado. Usa el botón **Detener** de Colab para desconectar el worker. Si la sesión gratuita se interrumpe, vuelve a abrir el notebook y ejecuta sus celdas otra vez.


In [ ]:
%pip install -q beautifulsoup4 boto3 requests gradio_client "git+https://github.com/javiertarazon/Scrapegraph-ai.git@c75c8084fae2d4f5ba01a8c218bc1168b67e3569"

In [ ]:
import os

try:
    from google.colab import userdata
except ImportError:  # Ejecucion local o en CI: los secretos vienen del entorno.
    userdata = None


def _read_secret(name):
    """Lee un secreto de Colab y, si no estamos en Colab, de las variables de entorno."""
    if userdata is not None:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
        if value:
            return value
    return os.environ.get(name)

REQUIRED = (
    "API_BASE_URL", "WORKER_API_TOKEN", "SCRIPT_SPACE_URL", "VOICE_STUDIO_URL",
    "HF_TOKEN", "S3_ENDPOINT_URL", "S3_BUCKET",
    "S3_ACCESS_KEY_ID", "S3_SECRET_ACCESS_KEY",
)
for name in REQUIRED:
    value = _read_secret(name)
    if not value:
        raise RuntimeError(
            f"Configura el secreto {name} en Colab o exporta la variable de entorno {name}"
        )
    os.environ[name] = value
    os.environ[name] = value
for name in ("YOUTUBE_API_KEY", "YOUTUBE_REGION_CODE"):
    value = _read_secret(name)  # Secret opcional no creado o no compartido con este notebook.
    if value:
        os.environ[name] = value
os.environ.setdefault("YOUTUBE_REGION_CODE", "VE")
for name in ("API_BASE_URL", "SCRIPT_SPACE_URL", "VOICE_STUDIO_URL"):
    os.environ[name] = os.environ[name].rstrip("/")
if not os.environ["API_BASE_URL"].startswith("https://"):
    raise RuntimeError("API_BASE_URL debe ser HTTPS y accesible desde Colab")
API_HEADERS = {"Authorization": "Bearer " + os.environ["WORKER_API_TOKEN"]}

In [ ]:
from pathlib import Path
import os
import re
from datetime import datetime, timedelta, timezone
from urllib.parse import urljoin, urlparse
import ipaddress
import inspect
import socket
import boto3
import requests
import threading
import tempfile
import time
import uuid
from bs4 import BeautifulSoup
from gradio_client import Client
from ddgs import DDGS
from ddgs.exceptions import DDGSException
from scrapegraphai.utils.research_web import ResearchWebError, search_on_web


PLATFORM_SEARCH = {
    "youtube": {"label": "YouTube", "domains": ("youtube.com",)},
    "reddit": {"label": "Reddit", "domains": ("reddit.com",)},
    "x": {"label": "X", "domains": ("x.com", "twitter.com"), "query": "(site:x.com OR site:twitter.com)"},
    "instagram": {"label": "Instagram", "domains": ("instagram.com",)},
    "tiktok": {"label": "TikTok", "domains": ("tiktok.com",)},
    "facebook": {"label": "Facebook", "domains": ("facebook.com",)},
}
MAX_RESULTS_PER_PLATFORM = 2
MAX_SOCIAL_TEXT_CHARS = 1200
WORKER_ID = uuid.uuid4().hex
WORKER_TYPE = "research"


HF_CLIENT_TOKEN_PARAM = "token" if "token" in inspect.signature(Client).parameters else "hf_token"

def hf_space_client(space_url):
    return Client(space_url, **{HF_CLIENT_TOKEN_PARAM: os.environ["HF_TOKEN"]}, verbose=False)


def search_youtube_videos(topic, region_code="VE"):
    api_key = os.environ["YOUTUBE_API_KEY"]
    now = datetime.now(timezone.utc)
    published_after = (now - timedelta(days=14)).isoformat().replace("+00:00", "Z")
    search = requests.get(
        "https://www.googleapis.com/youtube/v3/search",
        params={"part": "snippet", "type": "video", "order": "viewCount",
                "q": topic, "maxResults": 5, "regionCode": region_code,
                "publishedAfter": published_after, "key": api_key},
        timeout=(10, 30),
    )
    search.raise_for_status()
    ids = [item.get("id", {}).get("videoId") for item in search.json().get("items", [])]
    ids = [video_id for video_id in ids if video_id]
    if not ids:
        return [], {"platform": "youtube", "status": "no_results", "result_count": 0,
                    "message": "La API no encontró videos coincidentes de los últimos 14 días."}
    details = requests.get(
        "https://www.googleapis.com/youtube/v3/videos",
        params={"part": "snippet,statistics", "id": ",".join(ids), "key": api_key},
        timeout=(10, 30),
    )
    details.raise_for_status()
    sources = []
    for item in details.json().get("items", []):
        snippet = item.get("snippet", {})
        stats = item.get("statistics", {})
        published_at = snippet.get("publishedAt")
        try:
            published = datetime.fromisoformat(published_at.replace("Z", "+00:00"))
            age_hours = max((now - published).total_seconds() / 3600, 1 / 60)
            views = int(stats.get("viewCount", 0))
        except (AttributeError, TypeError, ValueError):
            continue
        title = snippet.get("title", "")
        description = snippet.get("description", "")
        sources.append({
            "platform": "youtube", "source_type": "video_metrics",
            "title": title, "url": f"https://www.youtube.com/watch?v={item['id']}",
            "text": f"{title}\n{description[:1000]}",
            "view_count": views, "views_per_hour": round(views / age_hours, 2),
            "published_at": published_at, "channel": snippet.get("channelTitle", ""),
        })
    sources.sort(key=lambda item: item["views_per_hour"], reverse=True)
    return sources, {
        "platform": "youtube", "status": "metrics_available", "result_count": len(sources),
        "message": "Ordenado por vistas por hora entre resultados coincidentes recientes; no representa un ranking global de viralidad.",
    }


def validate_public_url(url):
    parsed = urlparse(url)
    if parsed.scheme not in ("http", "https") or not parsed.hostname:
        raise ValueError("source_url debe ser una URL HTTP/HTTPS valida")
    addresses = socket.getaddrinfo(parsed.hostname, parsed.port or (443 if parsed.scheme == "https" else 80))
    if not addresses or any(not ipaddress.ip_address(item[4][0].split("%")[0]).is_global for item in addresses):
        raise ValueError("source_url debe apuntar a un servidor publico")
    return parsed


def extract_article(source_url):
    current_url = source_url
    for _ in range(6):
        validate_public_url(current_url)
        response = requests.get(
            current_url,
            headers={"User-Agent": "MarketingContentOS/1.0 (article research)"},
            timeout=(10, 30),
            allow_redirects=False,
        )
        if response.is_redirect:
            location = response.headers.get("Location")
            if not location:
                raise RuntimeError("La fuente devolvio una redireccion sin destino")
            current_url = urljoin(current_url, location)
            continue
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        for element in soup(("script", "style", "noscript", "svg", "nav", "footer")):
            element.decompose()
        title_tag = soup.find("meta", attrs={"property": "og:title"})
        description_tag = soup.find("meta", attrs={"property": "og:description"}) or soup.find("meta", attrs={"name": "description"})
        title = (title_tag.get("content", "").strip() if title_tag else "") or (soup.title.get_text(" ", strip=True) if soup.title else "")
        text = soup.get_text(" ", strip=True)
        description = description_tag.get("content", "").strip() if description_tag else ""
        if description and description not in text:
            text = f"{text} {description}".strip()
        if len(text) < 100:
            raise RuntimeError("No se pudo extraer suficiente texto de la fuente")
        return {"url": current_url, "title": title, "text": text[:18000]}
    raise RuntimeError("La fuente excedio el maximo de redirecciones")


def matches_platform(url, platform):
    hostname = (urlparse(url).hostname or "").lower().rstrip(".")
    allowed_hosts = {
        host
        for domain in PLATFORM_SEARCH[platform]["domains"]
        for host in (domain, f"www.{domain}", f"m.{domain}")
    }
    return hostname in allowed_hosts


def search_public_platform(topic, platform):
    if platform not in PLATFORM_SEARCH:
        raise ValueError(f"Plataforma de investigación no admitida: {platform}")
    search_config = PLATFORM_SEARCH[platform]
    if platform == "youtube" and os.environ.get("YOUTUBE_API_KEY"):
        try:
            videos, api_status = search_youtube_videos(
                topic, os.environ.get("YOUTUBE_REGION_CODE", "VE").upper()
            )
            if videos:
                return videos, api_status
        except (requests.RequestException, KeyError, TypeError, ValueError):
            # Never print the request exception: it may contain the API key URL.
            pass
    search_query = f"{search_config.get('query', 'site:' + search_config['domains'][0])} {topic}"
    search_error = ""
    try:
        urls = search_on_web(
            search_query,
            search_engine="duckduckgo",
            max_results=MAX_RESULTS_PER_PLATFORM,
            language="es",
        )
    except (ResearchWebError, TimeoutError, ImportError, ValueError) as error:
        urls = []
        search_error = str(error)[:240]

    sources = []
    errors = []
    for url in dict.fromkeys(urls):
        if not matches_platform(url, platform):
            continue
        try:
            article = extract_article(url)
        except (requests.RequestException, RuntimeError, ValueError, OSError) as error:
            errors.append(str(error)[:120])
            continue
        article["platform"] = platform
        article["source_type"] = "page"
        article["text"] = article["text"][:MAX_SOCIAL_TEXT_CHARS]
        sources.append(article)
        if len(sources) >= MAX_RESULTS_PER_PLATFORM:
            break

    if not sources:
        try:
            snippets = DDGS().text(search_query, max_results=MAX_RESULTS_PER_PLATFORM)
        except (DDGSException, TimeoutError, OSError) as error:
            errors.append(f"DuckDuckGo snippets: {str(error)[:120]}")
            snippets = []
        for snippet in snippets:
            url = snippet.get("href", "")
            text = snippet.get("body", "").strip()
            if not url or not matches_platform(url, platform) or len(text) < 40:
                continue
            sources.append({
                "platform": platform,
                "source_type": "search_snippet",
                "url": url,
                "title": snippet.get("title", "").strip(),
                "text": text[:MAX_SOCIAL_TEXT_CHARS],
            })
            if len(sources) >= MAX_RESULTS_PER_PLATFORM:
                break

    if sources and any(item["source_type"] == "page" for item in sources):
        status = "ok"
        message = ""
    elif sources:
        status = "snippet_only"
        message = "Se usan solo fragmentos del índice de DuckDuckGo; no se pudo leer el post completo."
    elif urls:
        status = "no_public_content"
        message = "; ".join(errors[:2]) or "Los resultados no eran páginas públicas de esta plataforma."
    elif search_error:
        status = "search_error"
        message = search_error
    else:
        status = "no_results"
        message = "DuckDuckGo no devolvió resultados para la búsqueda pública."
    return sources, {
        "platform": platform,
        "status": status,
        "result_count": len(sources),
        "message": message,
    }


ACTIVE_STAGE = "idle"


def report_worker_status(current_job_id=None, stage="idle", stopped=False):
    global ACTIVE_STAGE
    ACTIVE_STAGE = "stopped" if stopped else stage
    response = requests.post(
        f"{os.environ['API_BASE_URL']}/v1/workers/heartbeat",
        headers=API_HEADERS,
        json={
            "worker_id": WORKER_ID,
            "worker": WORKER_TYPE,
            "current_job_id": str(current_job_id) if current_job_id else None,
            "stage": ACTIVE_STAGE,
            "stopped": stopped,
        },
        timeout=20,
    )
    response.raise_for_status()


def start_lease_heartbeat(job_id, lease_headers):
    stop = threading.Event()
    lease_lost = threading.Event()

    def renew_lease():
        while not stop.wait(60):
            try:
                response = requests.post(
                    f"{os.environ['API_BASE_URL']}/v1/jobs/{job_id}/heartbeat",
                    headers=lease_headers,
                    timeout=30,
                )
                response.raise_for_status()
            except requests.RequestException:
                lease_lost.set()
                print(f"No se pudo renovar el lease del trabajo {job_id}; se detendra en el siguiente punto seguro.")
                return
            try:
                report_worker_status(job_id, ACTIVE_STAGE)
            except requests.RequestException as error:
                print(f"No se pudo actualizar la presencia del worker: {error}")

    thread = threading.Thread(target=renew_lease, daemon=True)
    thread.start()
    return stop, thread, lease_lost


def run_worker_loop(process_one, label, poll_seconds=20):
    print(f"Worker {label} conectado. La celda seguirá esperando trabajos; usa Detener para desconectarlo.")
    last_wait_notice = 0.0
    while True:
        try:
            report_worker_status(None, "idle")
            processed = process_one()
            if processed:
                report_worker_status(None, "idle")
                last_wait_notice = 0.0
                continue
            now = time.monotonic()
            if now - last_wait_notice >= 300:
                print(f"Worker {label} activo; sin trabajos pendientes. Reintentando cada {poll_seconds} s.")
                last_wait_notice = now
            time.sleep(poll_seconds)
        except KeyboardInterrupt:
            try:
                report_worker_status(None, "stopped", stopped=True)
            except requests.RequestException:
                pass
            print(f"Worker {label} detenido. Puedes volver a ejecutar esta celda para reconectarlo.")
            break
        except Exception as error:
            print(f"Error del worker {label}: {error}. Se reintentará en 30 s.")
            try:
                report_worker_status(None, "idle")
            except requests.RequestException:
                pass
            time.sleep(30)


def ensure_active_lease(lease_lost):
    if lease_lost.is_set():
        raise RuntimeError("El lease del trabajo se perdió; no se enviará el resultado tardío.")


def report_progress(job_id, lease_headers, stage, message, percent):
    response = requests.post(
        f"{os.environ['API_BASE_URL']}/v1/jobs/{job_id}/progress",
        headers=lease_headers,
        json={"stage": stage, "message": message, "percent": percent},
        timeout=30,
    )
    response.raise_for_status()
    try:
        report_worker_status(job_id, stage)
    except requests.RequestException as error:
        print(f"No se pudo actualizar la etapa del worker: {error}")


def process_one_research_job():
    response = requests.post(
        f"{os.environ['API_BASE_URL']}/v1/jobs/next",
        params={"worker": "research"}, headers=API_HEADERS, timeout=30,
    )
    response.raise_for_status()
    job = response.json()
    if job is None:
        print("No hay investigaciones pendientes.")
        return False

    job_id = job["job_id"]
    try:
        report_worker_status(job_id, "starting")
    except requests.RequestException as error:
        print(f"No se pudo actualizar la presencia del worker: {error}")
    lease_headers = {**API_HEADERS, "X-Worker-Lease": job["lease_token"]}
    heartbeat_stop, heartbeat_thread, lease_lost = start_lease_heartbeat(job_id, lease_headers)
    reference_path = None
    try:
        request = job["request"]
        source_url = request.get("source_url")
        research_sources = []
        platform_status = []
        if source_url:
            report_progress(job_id, lease_headers, "research_source", "Extrayendo la URL de referencia", 10)
            article = extract_article(source_url)
            article["platform"] = "web"
            article["text"] = article["text"][:4000]
            research_sources.append(article)
            platform_status.append({"platform": "web", "status": "ok", "result_count": 1, "message": ""})
        selected_platforms = request.get("platforms") or (list(PLATFORM_SEARCH) if not source_url else [])
        for index, platform in enumerate(selected_platforms):
            label = PLATFORM_SEARCH.get(platform, {}).get("label", platform)
            progress = 10 + int(20 * index / max(1, len(selected_platforms)))
            report_progress(
                job_id, lease_headers, "research_search",
                f"Buscando contenido público en {label}", progress,
            )
            found_sources, status = search_public_platform(request["topic"], platform)
            research_sources.extend(found_sources)
            platform_status.append(status)
        if not research_sources:
            detail = "; ".join(
                f"{item['platform']}: {item['status']} - {item['message']}"
                for item in platform_status
            )
            raise RuntimeError(
                "No se pudo extraer contenido público. "
                "Las redes pueden no estar indexadas o bloquear el acceso anónimo. " + detail
            )
        report_progress(job_id, lease_headers, "research_script", "Analizando fuentes y generando guion", 32)
        source_context = "\n\n".join(
            f"Plataforma: {item['platform']}\nTipo de fuente: {item.get('source_type', 'page')}\nTitulo: {item['title']}\n"
            f"URL: {item['url']}\nContenido: {item['text']}"
            for item in research_sources
        )
        prompt = (
            "Eres un estratega de contenido y creador de videos virales. Analiza las fuentes publicas recopiladas sobre el tema.\n"
            "Identifica patrones de alto impacto: gancho inicial (hook), formato, ritmo y palabras clave observables en las fuentes.\n"
            "Si hay metricas de YouTube, compara vistas por hora y fecha; no llames viral a un video solo por tener muchas vistas.\n"
            "Separa hechos citados, señales cuantitativas y recomendaciones; no inventes tendencias, datos ni citas.\n"
            "Crea una idea y un guion originales inspirados en patrones, sin copiar frases ni guiones de las fuentes.\n"
            "Devuelve exactamente dos secciones en espanol con estos encabezados, sin formato Markdown ni texto adicional:\n"
            "INVESTIGACION:\n(Gancho detectado, palabras clave, idea central e investigacion)\n"
            "GUION:\n(Solo la narracion hablada, empezando con un hook demoledor en los primeros 3 segundos, directa y dinamica; no excedas aproximadamente "
            f"{request['duration_seconds']} segundos)\n\n"
            f"Tema: {request['topic']}\nEstilo: {request['style']}\n"
            f"Fuentes públicas recopiladas:\n{source_context}"
        )
        ensure_active_lease(lease_lost)
        script_client = hf_space_client(os.environ["SCRIPT_SPACE_URL"])
        generated = script_client.submit(prompt, api_name="/generate").result(timeout=900)
        if not isinstance(generated, str):
            raise RuntimeError("El Space de guiones devolvio un formato inesperado")
        sections = re.fullmatch(
            r"\s*INVESTIGACION\s*:\s*(.*?)\s*GUION\s*:\s*(.*?)\s*",
            generated, flags=re.IGNORECASE | re.DOTALL,
        )
        if not sections:
            raise RuntimeError("El Space no devolvio las secciones de investigacion y guion esperadas")
        research_text, script = (part.strip() for part in sections.groups())
        if not research_text or not script:
            raise RuntimeError("El Space devolvio investigacion o guion vacios")

        ensure_active_lease(lease_lost)
        report_progress(job_id, lease_headers, "voice_synthesis", "Preparando narración de voz", 55)
        storage = boto3.client(
            "s3", endpoint_url=os.environ["S3_ENDPOINT_URL"], region_name="auto",
            aws_access_key_id=os.environ["S3_ACCESS_KEY_ID"],
            aws_secret_access_key=os.environ["S3_SECRET_ACCESS_KEY"],
        )
        voice_reference = request.get("voice_reference")
        reference_transcript = ""
        if voice_reference:
            reference_path = Path(tempfile.gettempdir()) / f"{job_id}-voice-reference.wav"
            storage.download_file(os.environ["S3_BUCKET"], voice_reference["object_key"], str(reference_path))
            reference_transcript = voice_reference["transcript"]
        voice_client = hf_space_client(os.environ["VOICE_STUDIO_URL"])
        voice_path = voice_client.submit(
            script,
            request.get("voice_gender", "female"),
            request.get("voice_age", "young adult"),
            request.get("voice_pitch", "moderate pitch"),
            str(reference_path) if reference_path else None,
            reference_transcript,
            api_name="/predict",
        ).result(timeout=1800)
        if not isinstance(voice_path, str) or not Path(voice_path).is_file():
            raise RuntimeError("El Space privado no devolvio el archivo de audio")
        voice_bytes = Path(voice_path).read_bytes()
        if not voice_bytes.startswith(b"RIFF"):
            raise RuntimeError("VoiceStudio devolvio un archivo que no es WAV")

        report_progress(job_id, lease_headers, "audio_storage", "Guardando narración en R2 privado", 85)
        audio_key = f"audio/{job_id}.wav"
        storage.put_object(Bucket=os.environ["S3_BUCKET"], Key=audio_key,
                           Body=voice_bytes, ContentType="audio/wav")
        ensure_active_lease(lease_lost)
        complete = requests.post(
            f"{os.environ['API_BASE_URL']}/v1/jobs/{job_id}/complete",
            headers=lease_headers,
            json={"output": {"research": {"source": research_sources[0]["url"], "notes": research_text, "sources": [{"platform": item["platform"], "source_type": item.get("source_type", "page"), "title": item["title"], "url": item["url"], "view_count": item.get("view_count"), "views_per_hour": item.get("views_per_hour"), "published_at": item.get("published_at"), "channel": item.get("channel")} for item in research_sources], "platform_status": platform_status}, "script": script, "audio_object_key": audio_key}},
            timeout=60,
        )
        complete.raise_for_status()
        print(f"Investigacion y voz listas; se encolo el render: {job_id}")
    except Exception as error:
        try:
            failed = requests.post(
                f"{os.environ['API_BASE_URL']}/v1/jobs/{job_id}/fail",
                headers=lease_headers, json={"error": str(error)[:1900]}, timeout=60,
            )
            failed.raise_for_status()
        except requests.RequestException:
            print("No se pudo informar el fallo; la API recuperará el trabajo al vencer el lease.")
        raise RuntimeError(f"Trabajo {job_id} no completado: {error}") from error
    finally:
        heartbeat_stop.set()
        heartbeat_thread.join(timeout=5)
        if reference_path and reference_path.exists():
            reference_path.unlink()
    return True


# Mantén esta celda ejecutándose: el dashboard local encola los trabajos y este worker
# los procesa en orden. Detén la celda cuando quieras desconectar esta sesión de Colab.
run_worker_loop(process_one_research_job, "investigación y voz")
